# Giai đoạn 3: Baselines cho **gợi ý phim theo nội dung văn bản** (Content-based IR)

## Bài toán (CineSense)

- **Input:** corpus phim, mỗi phim có văn bản từ **review** (ưu tiên) và/hoặc **metadata** (overview, genre, title).
- **Output cho offline eval:** với mỗi phim *query*, xếp hạng phim *candidate* theo độ tương đồng vector → đo Precision@K, Recall@K, nDCG@K.
- **Nhãn yếu (weak supervision):** hai phim được coi *relevant* nếu có **≥1 genre trùng** (có thể chỉnh `MIN_GENRE_OVERLAP`).
- **Text dùng để vector hoá:** ưu tiên `review_profile`; nếu quá ngắn thì dùng `movie_profile` (cùng logic với artifact runtime hiện tại trong `api/recommender.py`).

## Đầu ra bắt buộc cho runtime

Sau khi chạy xong, notebook sẽ ghi:

- `training/artifacts/tfidf_latest/` — `metadata.json`, `movie_index.json`, `similar_by_movie.json` (TF-IDF tốt nhất trong grid).
- `training/artifacts/word2vec_latest/` — tương tự (Word2Vec tốt nhất).
- `eval_results.json` — cho notebook 05.

**UUID phim:** `uuid5(URL, "cinesense:core_movie:{tmdb_id}")` — ổn định giữa các lần train, đồng bộ với `scripts/seed_sqlite_from_csv.py` khi bạn dùng `movie_index.json` này.

## Tuỳ chọn: SBERT

Nếu cài `sentence-transformers`, bật `RUN_SBERT = True` trong cell code để export `training/artifacts/sbert_latest/` (`embeddings.npy` + `similar_by_movie.json`). Base model mặc định lấy từ biến môi trường `EMBEDDING_MODEL` hoặc `sentence-transformers/all-MiniLM-L6-v2`.

> Nếu muốn dùng **English fine-tuned bi-encoder** cho runtime hiện tại, hãy chạy thêm:
> `python -m Notebook_Report.retrieval.build_query_bank`
> `python -m Notebook_Report.retrieval.finetune_biencoder`

## Trước khi chạy

1. `Notebook_Report/02_Data_Preprocessing_EDA.ipynb` → có `cleaned_profiles.csv`.
2. `cinesense_reviews.csv` (cùng thư mục) để đếm `review_count` trong `movie_index`.



In [1]:
# =============================================================================
# Giai đoạn 3 — Train / eval baseline + export artifact
# =============================================================================
from __future__ import annotations

import json
import math
import os
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from uuid import NAMESPACE_URL, uuid5

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Cấu hình ---
MIN_GENRE_OVERLAP = 1
QUERY_RATIO = 0.2
NEIGHBOR_K = 20
K_EVAL = (5, 10)
SEED = 42

# Semantic embedding (tuỳ chọn)
RUN_SBERT = os.getenv("RUN_SBERT_IN_NB", "1") == "1"
SBERT_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
SBERT_BATCH = int(os.getenv("SBERT_BATCH", "32"))

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "Notebook_Report":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "Notebook_Report"
ARTIFACT_ROOT = NOTEBOOK_DIR / "training" / "artifacts"


def _stable_movie_id(tmdb_id: int) -> str:
    return str(uuid5(NAMESPACE_URL, f"cinesense:core_movie:{int(tmdb_id)}"))


def retrieval_text(row: pd.Series) -> str:
    """Ưu tiên review_profile; nếu quá ngắn thì fallback sang movie_profile hoặc title+genres như artifact runtime."""
    rp = str(row.get("review_profile", "")).strip()
    if len(rp) >= 24:
        return rp[:8000]
    mp = str(row.get("movie_profile", "")).strip()
    if len(mp) >= 8:
        return mp[:8000]
    t = str(row.get("title", ""))
    g = str(row.get("genres", ""))
    return f"{t} | {g}"


def precision_at_k(predicted: list[int], relevant: set[int], k: int) -> float:
    if k <= 0:
        return 0.0
    top_k = predicted[:k]
    if not top_k:
        return 0.0
    return sum(1 for idx in top_k if idx in relevant) / k


def recall_at_k(predicted: list[int], relevant: set[int], k: int) -> float:
    if not relevant:
        return 0.0
    top_k = predicted[:k]
    hit_count = sum(1 for idx in top_k if idx in relevant)
    return hit_count / len(relevant)


def ndcg_at_k(predicted: list[int], relevant: set[int], k: int) -> float:
    if k <= 0 or not relevant:
        return 0.0
    top_k = predicted[:k]
    dcg = 0.0
    for rank, idx in enumerate(top_k, start=1):
        if idx in relevant:
            dcg += 1.0 / math.log2(rank + 1)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(r + 1) for r in range(1, ideal_hits + 1))
    return (dcg / idcg) if idcg > 0 else 0.0


def build_genre_relevance(df: pd.DataFrame, min_overlap: int = 1) -> dict[int, set[int]]:
    genres_list: list[set[str]] = []
    for g in df["genres"].fillna(""):
        parts = [p.strip().lower() for p in str(g).split(",") if p.strip()]
        genres_list.append(set(parts))

    gold: dict[int, set[int]] = {}
    for i, gi in enumerate(genres_list):
        if not gi:
            gold[i] = set()
            continue
        rel = set()
        for j, gj in enumerate(genres_list):
            if i == j:
                continue
            if len(gi.intersection(gj)) >= min_overlap:
                rel.add(j)
        gold[i] = rel
    return gold


def split_query_candidate(n: int, query_ratio: float, seed: int) -> tuple[list[int], list[int]]:
    rng = random.Random(seed)
    indices = list(range(n))
    rng.shuffle(indices)
    qn = max(1, int(round(n * query_ratio)))
    query = indices[:qn]
    cand = indices[qn:]
    if not cand:
        cand = query[-1:]
        query = query[:-1]
    return query, cand


def topk_from_similarity(
    sim_row: np.ndarray, exclude_self: int, candidates: list[int], top_k: int
) -> list[int]:
    scores = [(j, float(sim_row[j])) for j in candidates if j != exclude_self]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [j for j, _ in scores[:top_k]]


def evaluate_model(
    sim: np.ndarray,
    query_ids: list[int],
    candidate_ids: list[int],
    gold: dict[int, set[int]],
    k_values: tuple[int, ...],
) -> dict[str, Any]:
    out: dict[str, Any] = {}
    for k in k_values:
        ps, rs, ns = [], [], []
        for q in query_ids:
            pred = topk_from_similarity(sim[q], q, candidate_ids, top_k=k)
            rel = gold.get(q, set())
            ps.append(precision_at_k(pred, rel, k))
            rs.append(recall_at_k(pred, rel, k))
            ns.append(ndcg_at_k(pred, rel, k))
        out[f"precision@{k}"] = float(np.mean(ps)) if ps else 0.0
        out[f"recall@{k}"] = float(np.mean(rs)) if rs else 0.0
        out[f"ndcg@{k}"] = float(np.mean(ns)) if ns else 0.0
    return out


def load_review_counts() -> dict[int, int]:
    p = NOTEBOOK_DIR / "cinesense_reviews.csv"
    if not p.exists():
        return {}
    r = pd.read_csv(p, usecols=["tmdb_id"])
    vc = r.groupby("tmdb_id").size()
    return {int(k): int(v) for k, v in vc.items()}


def build_movie_index_records(df: pd.DataFrame, review_counts: dict[int, int]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for _, r in df.iterrows():
        tid = int(r["tmdb_id"])
        genres = [g.strip() for g in str(r.get("genres", "")).split(",") if g.strip()]
        rd = str(r.get("release_date", "") or "")[:10]
        year = 0
        if len(rd) >= 4 and rd[:4].isdigit():
            year = int(rd[:4])
        rows.append(
            {
                "id": _stable_movie_id(tid),
                "tmdb_id": tid,
                "title": str(r.get("title", "") or ""),
                "overview": str(r.get("overview", "") or ""),
                "poster_path": str(r.get("poster_path", "") or ""),
                "genres": genres,
                "review_count": int(review_counts.get(tid, 0)),
                "release_year": year,
            }
        )
    return rows


def similar_by_movie_from_sim(
    sim: np.ndarray,
    df: pd.DataFrame,
    top_k: int,
) -> dict[str, list[dict[str, Any]]]:
    n = len(df)
    ids = [_stable_movie_id(int(df.iloc[i]["tmdb_id"])) for i in range(n)]
    out: dict[str, list[dict[str, Any]]] = {}
    for i in range(n):
        row = sim[i].copy()
        row[i] = -1.0
        nn = np.argsort(-row)[:top_k]
        out[ids[i]] = [{"movie_id": ids[j], "score": float(sim[i, j])} for j in nn]
    return out


def export_artifact(
    name: str,
    sim: np.ndarray,
    df: pd.DataFrame,
    extra_meta: dict[str, Any],
) -> None:
    out_dir = ARTIFACT_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    idx = build_movie_index_records(df, load_review_counts())
    out_dir.joinpath("movie_index.json").write_text(
        json.dumps(idx, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    sim_json = similar_by_movie_from_sim(sim, df, NEIGHBOR_K)
    out_dir.joinpath("similar_by_movie.json").write_text(
        json.dumps(sim_json, ensure_ascii=False), encoding="utf-8"
    )
    meta = {
        "artifact_type": extra_meta.get("artifact_type", "tfidf"),
        "artifact_version": name,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "movie_count": len(df),
        "top_k": NEIGHBOR_K,
        **{k: v for k, v in extra_meta.items() if k != "artifact_type"},
    }
    out_dir.joinpath("metadata.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"  -> Đã export: {out_dir}")


# --- Main ---
try:
    csv_path = NOTEBOOK_DIR / "cleaned_profiles.csv"
    df = pd.read_csv(csv_path).fillna("")
    print(f"Đã tải {len(df)} phim từ {csv_path.name}\n")

    df["_retrieval_text"] = df.apply(retrieval_text, axis=1)
    print("Cột vector hoá: `_retrieval_text` (review-first, fallback movie_profile/title+genres).")

    gold = build_genre_relevance(df, min_overlap=MIN_GENRE_OVERLAP)
    query_ids, candidate_ids = split_query_candidate(len(df), QUERY_RATIO, SEED)
    print(f"Eval split: query={len(query_ids)}, candidate={len(candidate_ids)} (seed={SEED})\n")

    tfidf_grid = [
        {"name": "tfidf_unigram", "ngram_range": (1, 1), "min_df": 1, "max_df": 1.0},
        {"name": "tfidf_uni_bigram", "ngram_range": (1, 2), "min_df": 1, "max_df": 1.0},
        {"name": "tfidf_uni_bigram_min2", "ngram_range": (1, 2), "min_df": 2, "max_df": 0.9},
    ]

    configs: list[dict[str, Any]] = []
    best_tfidf: tuple[float, np.ndarray, dict[str, Any], str] | None = None

    print("=== TF-IDF (grid) ===")
    for cfg in tfidf_grid:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=cfg["ngram_range"],
            min_df=cfg["min_df"],
            max_df=cfg["max_df"],
        )
        X = vectorizer.fit_transform(df["_retrieval_text"].astype(str).tolist())
        sim = cosine_similarity(X, X)
        metrics = evaluate_model(sim, query_ids, candidate_ids, gold, k_values=K_EVAL)
        row = {"model": cfg["name"], **metrics}
        configs.append({"model": cfg["name"], "type": "tfidf", "params": cfg, "metrics": metrics})
        print(cfg["name"], json.dumps(row, ensure_ascii=False, indent=2))
        score = metrics.get("ndcg@10", 0.0)
        if best_tfidf is None or score > best_tfidf[0]:
            best_tfidf = (score, sim, cfg, cfg["name"])

    assert best_tfidf is not None
    print("\n=== Export TF-IDF artifact (best theo nDCG@10) ===")
    print(best_tfidf[3], "ndcg@10=", best_tfidf[0])
    export_artifact(
        "tfidf_latest",
        best_tfidf[1],
        df,
        {
            "artifact_type": "tfidf",
            "model_name": str(best_tfidf[2]["name"]),
            "tfidf_params": best_tfidf[2],
        },
    )

    w2v_grid = [
        {"name": "w2v_dim50", "vector_size": 50, "window": 5, "min_count": 1, "sg": 1},
        {"name": "w2v_dim100", "vector_size": 100, "window": 5, "min_count": 1, "sg": 1},
    ]

    best_w2v: tuple[float, np.ndarray, dict[str, Any], str] | None = None

    print("\n=== Word2Vec (grid) ===")
    try:
        from gensim.models import Word2Vec

        tokenized = [str(t).split() for t in df["_retrieval_text"].astype(str).tolist()]

        def average_embedding(tokens: list[str], model: Word2Vec, dim: int) -> np.ndarray:
            vecs = [model.wv[t] for t in tokens if t in model.wv]
            if not vecs:
                return np.zeros(dim, dtype=np.float32)
            return np.mean(vecs, axis=0)

        nw = min(4, (os.cpu_count() or 2))
        for cfg in w2v_grid:
            model = Word2Vec(
                sentences=tokenized,
                vector_size=cfg["vector_size"],
                window=cfg["window"],
                min_count=cfg["min_count"],
                workers=nw,
                sg=cfg["sg"],
                seed=SEED,
            )
            emb = np.vstack(
                [average_embedding(toks, model, dim=cfg["vector_size"]) for toks in tokenized]
            )
            sim = cosine_similarity(emb, emb)
            metrics = evaluate_model(sim, query_ids, candidate_ids, gold, k_values=K_EVAL)
            row = {"model": cfg["name"], **metrics}
            configs.append({"model": cfg["name"], "type": "word2vec", "params": cfg, "metrics": metrics})
            print(cfg["name"], json.dumps(row, ensure_ascii=False, indent=2))
            score = metrics.get("ndcg@10", 0.0)
            if best_w2v is None or score > best_w2v[0]:
                best_w2v = (score, sim, cfg, cfg["name"])

        if best_w2v:
            print("\n=== Export Word2Vec artifact ===")
            print(best_w2v[3], "ndcg@10=", best_w2v[0])
            export_artifact(
                "word2vec_latest",
                best_w2v[1],
                df,
                {
                    "artifact_type": "word2vec",
                    "model_name": str(best_w2v[2]["name"]),
                    "w2v_params": best_w2v[2],
                },
            )
    except ImportError:
        print("gensim chưa có — bỏ Word2Vec. pip install gensim")

    if RUN_SBERT:
        print("\n=== Sentence-Transformers (SBERT) ===")
        try:
            from sentence_transformers import SentenceTransformer

            st = SentenceTransformer(SBERT_MODEL)
            texts = df["_retrieval_text"].astype(str).tolist()
            emb = st.encode(
                texts,
                batch_size=SBERT_BATCH,
                show_progress_bar=True,
                normalize_embeddings=True,
            )
            emb = np.asarray(emb, dtype=np.float64)
            sim = cosine_similarity(emb, emb)
            metrics = evaluate_model(sim, query_ids, candidate_ids, gold, k_values=K_EVAL)
            configs.append(
                {
                    "model": "sbert_" + SBERT_MODEL.split("/")[-1].replace(".", "_"),
                    "type": "sentence_transformer",
                    "params": {"model": SBERT_MODEL},
                    "metrics": metrics,
                }
            )
            print(json.dumps(metrics, ensure_ascii=False, indent=2))
            out_dir = ARTIFACT_ROOT / "sbert_latest"
            out_dir.mkdir(parents=True, exist_ok=True)
            np.save(out_dir / "embeddings.npy", emb)
            export_artifact(
                "sbert_latest",
                sim,
                df,
                {
                    "artifact_type": "sentence_transformer",
                    "model_name": SBERT_MODEL,
                    "embedding_dim": int(emb.shape[1]),
                },
            )
        except ImportError:
            print("sentence-transformers chưa có — bỏ SBERT.")

    rows_out = []
    for c in configs:
        m = c["metrics"]
        rows_out.append(
            {
                "model": c["model"],
                "type": c.get("type", ""),
                "precision@5": m.get("precision@5", 0.0),
                "recall@5": m.get("recall@5", 0.0),
                "ndcg@5": m.get("ndcg@5", 0.0),
                "precision@10": m.get("precision@10", 0.0),
                "recall@10": m.get("recall@10", 0.0),
                "ndcg@10": m.get("ndcg@10", 0.0),
            }
        )
    summary = pd.DataFrame(rows_out).sort_values("ndcg@10", ascending=False)
    display(summary)

    out_eval = NOTEBOOK_DIR / "eval_results.json"
    out_eval.write_text(json.dumps(configs, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nĐã lưu {out_eval}")

except FileNotFoundError:
    print("Không thấy cleaned_profiles.csv — chạy notebook 02 trước (working directory = Notebook_Report).")


Đã tải 4797 phim từ cleaned_profiles.csv

Cột vector hoá: `_retrieval_text` (review-first, fallback movie_profile/title+genres).
Eval split: query=959, candidate=3838 (seed=42)

=== TF-IDF (grid) ===
tfidf_unigram {
  "model": "tfidf_unigram",
  "precision@5": 0.7662148070907195,
  "recall@5": 0.002166183291704929,
  "ndcg@5": 0.76977836651325,
  "precision@10": 0.7499478623566215,
  "recall@10": 0.004164986831495678,
  "ndcg@10": 0.7573061709908647
}
tfidf_uni_bigram {
  "model": "tfidf_uni_bigram",
  "precision@5": 0.8064650677789365,
  "recall@5": 0.0026341908634640067,
  "ndcg@5": 0.8110487667938813,
  "precision@10": 0.7991657977059436,
  "recall@10": 0.0052142903015441056,
  "ndcg@10": 0.804486580410991
}
tfidf_uni_bigram_min2 {
  "model": "tfidf_uni_bigram_min2",
  "precision@5": 0.8095933263816476,
  "recall@5": 0.0027215655152003112,
  "ndcg@5": 0.8144761290579747,
  "precision@10": 0.8047966631908238,
  "recall@10": 0.0053793009206017225,
  "ndcg@10": 0.8096391038252034
}

==

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


w2v_dim50 {
  "model": "w2v_dim50",
  "precision@5": 0.6805005213764338,
  "recall@5": 0.001799707212965545,
  "ndcg@5": 0.6867236908432196,
  "precision@10": 0.6591240875912407,
  "recall@10": 0.003463228778465367,
  "ndcg@10": 0.6697520861624493
}


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


w2v_dim100 {
  "model": "w2v_dim100",
  "precision@5": 0.6992700729927008,
  "recall@5": 0.0018021019569271185,
  "ndcg@5": 0.7082350655063223,
  "precision@10": 0.6759124087591242,
  "recall@10": 0.003457726901860129,
  "ndcg@10": 0.6887222343222262
}

=== Export Word2Vec artifact ===
w2v_dim100 ndcg@10= 0.6887222343222262
  -> Đã export: /Users/kotori/CineSen/Notebook_Report/training/artifacts/word2vec_latest

=== Sentence-Transformers (SBERT) ===


...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/150 [00:00<?, ?it/s]

{
  "precision@5": 0.7199165797705944,
  "recall@5": 0.0018483135758639462,
  "ndcg@5": 0.7270697785373008,
  "precision@10": 0.7004171011470282,
  "recall@10": 0.0035103831103943,
  "ndcg@10": 0.7108826018860994
}
  -> Đã export: /Users/kotori/CineSen/Notebook_Report/training/artifacts/sbert_latest


,model,type,precision@5,recall@5,ndcg@5,precision@10,recall@10,ndcg@10
2,tfidf_uni_bigram_min2,tfidf,0.809593,0.002722,0.814476,0.804797,0.005379,0.809639
1,tfidf_uni_bigram,tfidf,0.806465,0.002634,0.811049,0.799166,0.005214,0.804487
0,tfidf_unigram,tfidf,0.766215,0.002166,0.769778,0.749948,0.004165,0.757306
5,sbert_paraphrase-multilingual-MiniLM-L12-v2,sentence_transformer,0.719917,0.001848,0.727070,0.700417,0.003510,0.710883
4,w2v_dim100,word2vec,0.699270,0.001802,0.708235,0.675912,0.003458,0.688722
3,w2v_dim50,word2vec,0.680501,0.001800,0.686724,0.659124,0.003463,0.669752



Đã lưu /Users/kotori/CineSen/Notebook_Report/eval_results.json
